# Лабораторная работа № 1
## Распределённая пакетная обработка логов робототехнических комплексов в Apache Spark

**Дисциплина:** «Методы обработки больших данных»

**Направление:** 44.03.05 Педагогическое образование

**Профиль:** «Информатика и дополнительное образование (робототехника)»

**Этап конвейера:** `Collect → Store → Process`

**Среда:** Google Colab · Python 3 · PySpark 4.2.0 · режим `local[*]`

---

### Цель работы

Освоить пакетную обработку журнала робототехнической телеметрии средствами PySpark: пройти путь
от «ручного» MapReduce на чистом Python до DataFrame API и Spark SQL, исследовать ленивые
вычисления, партиционирование и shuffle.

### Что нужно сдать

1. Этот ноутбук с выполненными заданиями (`.ipynb`, все ячейки выполнены сверху вниз).
2. Ссылку на ноутбук в Colab либо в вашем GitHub-репозитории.
3. Заполненную форму **TEACH CARD** в конце ноутбука.

### Критерии оценки — ровно 10 баллов

| Часть | Содержание | Баллы |
|---|---|---|
| 1 | MapReduce «вручную» на Python + RDD API | 2 |
| 2 | Агрегация через DataFrame API, расчёт RMS | 2 |
| 3 | Эквивалентная аналитика через Spark SQL | 2 |
| 4 | Physical plan и эксперимент с партиционированием | 2 |
| 5 | Заполненная форма TEACH CARD | 2 |

> **Правила выполнения.** Не ищите готовые решения у однокурсников и в интернете — пользуйтесь
> `help()` и документацией. Сначала добейтесь работающего решения, оптимизацией занимайтесь потом.
> Все ячейки с `assert` — это самопроверка: если проверка проходит, задание засчитано.

### Источники

* Pierre Navaro. `big-data`: <https://github.com/pnavaro/big-data> — ноутбуки 04-WordCount, 05-MapReduce, 15-PySpark
* Giuseppe Tolomei. `big-data-computing`: <https://github.com/gtolomei/big-data-computing>
* Apache Spark. SQL Programming Guide: <https://spark.apache.org/docs/latest/sql-programming-guide>

> ### ВАРИАНТ ПРЕПОДАВАТЕЛЯ
> Ноутбук содержит полное решение всех заданий и выполнен в среде, эквивалентной Google Colab (PySpark 4.2.0, `local[*]`).
> Ячейки с решениями помечены в тексте; студенческий вариант отличается только содержимым кодовых ячеек.
> Численные значения времени зависят от runtime и у студентов будут другими — оценивается корректность метода, а не абсолютные цифры.


---
## Задание 0. Подготовка среды

Устанавливаем PySpark и создаём `SparkSession` в локальном режиме. В Colab установка занимает
около минуты и выполняется один раз на runtime.

In [1]:
!pip -q install "pyspark==4.0.0" pandas pyarrow
import pyspark; print("PySpark", pyspark.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 1.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
PySpark 4.0.0


In [2]:
import time
import re
from collections import Counter, defaultdict
from itertools import chain

from pyspark.sql import SparkSession, functions as F
from pyspark.storagelevel import StorageLevel

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Lab01_RobotBatchProcessing")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark version:      ", spark.version)
print("Default parallelism:", sc.defaultParallelism)

Spark version:       4.0.0
Default parallelism: 2


---
# Часть 1. MapReduce своими руками (2 балла)

Прежде чем запускать распределённый движок, полезно один раз собрать MapReduce вручную — на
обычном Python. Тогда становится видно, что Spark не делает никакой магии: он выполняет ровно
те же три шага, только на многих машинах.

Работать будем с журналом диагностических сообщений робота — это текстовые строки вида

```
2026-03-01T10:15:02 robot-3 WARN motor controller timeout
```

Классическая задача над таким журналом — подсчитать, какие сообщения встречаются чаще всего.
Это тот самый «Hello World» больших данных, word count.

## 1.1. Генерация журнала диагностики

Внешний датасет не нужен: журнал порождается детерминированно, поэтому результаты у всех
студентов совпадут.

In [4]:
import random

MESSAGES = [
    "motor controller timeout",
    "lidar packet dropped",
    "navigation goal aborted",
    "battery voltage low",
    "imu calibration drift",
    "wheel odometry mismatch",
    "camera frame skipped",
    "emergency stop released",
]
LEVELS = ["INFO", "WARN", "ERROR"]

random.seed(42)

def make_log(path, n_lines, n_robots=8):
    """Создаёт текстовый файл журнала из n_lines строк."""
    with open(path, "w", encoding="utf-8") as f:
        for i in range(n_lines):
            robot = f"robot-{i % n_robots}"
            level = random.choices(LEVELS, weights=[6, 3, 1])[0]
            msg = random.choices(MESSAGES, weights=[9, 7, 5, 5, 4, 3, 2, 1])[0]
            ts = f"2026-03-01T{(i // 3600) % 24:02d}:{(i // 60) % 60:02d}:{i % 60:02d}"
            f.write(f"{ts} {robot} {level} {msg}\n")

make_log("robot_log.txt", 200_000)

with open("robot_log.txt", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip())
print("Всего строк:", sum(1 for _ in open("robot_log.txt", encoding="utf-8")))

2026-03-01T00:00:00 robot-0 WARN motor controller timeout
2026-03-01T00:00:01 robot-1 INFO motor controller timeout
2026-03-01T00:00:02 robot-2 WARN battery voltage low
Всего строк: 200000


## Упражнение 1.1

Напишите функцию `mapper(path)`, которая читает файл журнала и возвращает **список пар**
`(слово, 1)` для всех слов в **тексте сообщения** — то есть начиная с четвёртого поля строки
(отметка времени, идентификатор робота и уровень в подсчёте не участвуют).

Ожидаемый результат:

```python
mapper("robot_log.txt")[:4]
[('motor', 1), ('controller', 1), ('timeout', 1), ('lidar', 1)]
```

In [5]:
def mapper(path):
    """Читает журнал и возвращает список пар (слово, 1)."""
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            fields = line.strip().split()
            for word in fields[3:]:          # первые три поля — ts, robot, level
                pairs.append((word, 1))
    return pairs


pairs = mapper("robot_log.txt")
print(pairs[:4])
print("Всего пар:", len(pairs))

assert isinstance(pairs, list) and isinstance(pairs[0], tuple)
assert pairs[0][1] == 1
assert len(pairs) > 40_000

[('motor', 1), ('controller', 1), ('timeout', 1), ('motor', 1)]
Всего пар: 600000


## Упражнение 1.2

Напишите функцию `partitioner(pairs)` — она моделирует фазу **shuffle**: группирует пары по
ключу и возвращает список `(слово, [1, 1, ..., 1])`, отсортированный по слову.

Подсказка: пригодится `collections.defaultdict(list)`.

```python
partitioner(mapper("robot_log.txt"))[:2]
[('aborted', [1, 1, ...]), ('battery', [1, 1, ...])]
```

In [8]:
def partitioner(pairs):
    """Группирует пары (слово, 1) по ключу. Возвращает [(слово, [1, 1, ...]), ...]."""
    buckets = defaultdict(list)
    for key, value in pairs:
        buckets[key].append(value)
    return sorted(buckets.items())


grouped = partitioner(pairs)
print(grouped[0][0], "->", len(grouped[0][1]), "вхождений")
print("Различных слов:", len(grouped))

assert grouped == sorted(grouped)
assert sum(len(v) for _, v in grouped) == len(pairs)

aborted -> 27813 вхождений
Различных слов: 24


## Упражнение 1.3

Напишите функцию `reducer(item)`, которая принимает пару `(слово, [1, 1, ...])` и возвращает
`(слово, количество)`. Затем примените её ко всем сгруппированным парам через встроенную
функцию `map` и выведите **пять самых частых слов**.

```python
reducer(('timeout', [1, 1, 1, 1, 1]))
('timeout', 5)
```

In [9]:
def reducer(item):
    """Свёртка: (слово, [1, 1, ...]) -> (слово, количество)."""
    key, values = item
    return (key, sum(values))


counts = dict(map(reducer, grouped))
top5 = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:5]
print("Топ-5 слов:", top5)

assert reducer(("timeout", [1, 1, 1, 1, 1])) == ("timeout", 5)
assert sum(counts.values()) == len(pairs)
assert counts == dict(Counter(w for w, _ in pairs))   # сверка с Counter

Топ-5 слов: [('controller', 49971), ('motor', 49971), ('timeout', 49971), ('dropped', 38636), ('lidar', 38636)]


> **Наблюдение.** Три функции, которые вы написали, — это и есть map, shuffle и reduce.
> `Counter` из стандартной библиотеки решает ту же задачу одной строкой, но не масштабируется:
> он требует, чтобы весь словарь поместился в память одной машины. Именно поэтому дальше
> появляется Spark.

## Упражнение 1.4. То же самое на RDD

Теперь повторите подсчёт средствами Spark. Цепочка операций:

1. `sc.textFile(...)` — прочитать файл в RDD;
2. `flatMap` — разбить строку на слова сообщения (отбросив первые три поля);
3. `map` — превратить слово в пару `(слово, 1)`;
4. `reduceByKey(add)` — просуммировать по ключу;
5. `takeOrdered(5, key=...)` — взять пять самых частых.

> При совпадении частот порядок неоднозначен, поэтому в ключе сортировки указывайте и само
> слово: `key=lambda kv: (-kv[1], kv[0])`.

Обратите внимание: шаги 2–4 — это **transformations**, они ленивые; вычисление запустится
только на шаге 5, который является **action**.

In [10]:
from operator import add

lines = sc.textFile("robot_log.txt")

word_counts = (
    lines
    .flatMap(lambda line: line.split()[3:])
    .map(lambda word: (word, 1))
    .reduceByKey(add)
)

top5_rdd = word_counts.takeOrdered(5, key=lambda kv: (-kv[1], kv[0]))
print("Топ-5 слов (Spark):", top5_rdd)

assert dict(word_counts.collect()) == counts, "Spark и чистый Python должны совпасть"
assert top5_rdd == top5

Топ-5 слов (Spark): [('controller', 49971), ('motor', 49971), ('timeout', 49971), ('dropped', 38636), ('lidar', 38636)]


**Вопрос 1.5 (ответьте текстом в следующей ячейке).**

1. Какая из операций цепочки — `flatMap`, `map`, `reduceByKey` — создаёт shuffle и почему?
2. Чем `reduceByKey` предпочтительнее связки `groupByKey().mapValues(sum)`?

In [11]:
ANSWER_1_5 = """
1. Shuffle создаёт reduceByKey. flatMap и map — узкие зависимости (narrow): каждая выходная
   partition вычисляется из одной входной, данные не покидают узел. reduceByKey должен собрать
   вместе все значения одного ключа, а они разбросаны по всем partition, поэтому требуется
   перераспределение записей по сети — широкая зависимость (wide) и граница stage.

2. reduceByKey выполняет частичную агрегацию на стороне map (combiner) до пересылки: с каждого
   узла уходит по одной паре на ключ вместо всех значений. groupByKey сначала гонит по сети все
   значения целиком и только потом суммирует — трафик на порядки больше, а при большом числе
   значений одного ключа возможна нехватка памяти на редьюсере.
"""
print(ANSWER_1_5)


1. Shuffle создаёт reduceByKey. flatMap и map — узкие зависимости (narrow): каждая выходная
   partition вычисляется из одной входной, данные не покидают узел. reduceByKey должен собрать
   вместе все значения одного ключа, а они разбросаны по всем partition, поэтому требуется
   перераспределение записей по сети — широкая зависимость (wide) и граница stage.

2. reduceByKey выполняет частичную агрегацию на стороне map (combiner) до пересылки: с каждого
   узла уходит по одной паре на ключ вместо всех значений. groupByKey сначала гонит по сети все
   значения целиком и только потом суммирует — трафик на порядки больше, а при большом числе
   значений одного ключа возможна нехватка памяти на редьюсере.



---
# Часть 2. DataFrame API (2 балла)

Переходим к структурированным данным. Сгенерируем журнал телеметрии на 500 000 записей:
восемь роботов, показания акселерометра по трём осям, температура двигателя и напряжение
батареи. Внешний датасет по-прежнему не нужен.

In [13]:
N = 500_000

logs = (
    spark.range(N)
    .withColumn("robot_id",   F.concat(F.lit("robot-"), (F.col("id") % 8).cast("string")))
    .withColumn("sensor_id",  F.concat(F.lit("imu-"), (F.col("id") % 4).cast("string")))
    .withColumn("ts_ms",      (F.lit(1_720_000_000_000) + F.col("id") * 50).cast("long"))
    .withColumn("ax",         F.sin(F.col("id") / 25.0) + ((F.col("id") % 7) - 3) * 0.01)
    .withColumn("ay",         F.cos(F.col("id") / 40.0) + ((F.col("id") % 5) - 2) * 0.01)
    .withColumn("az",         F.lit(9.81) + F.sin(F.col("id") / 60.0) * 0.08)
    .withColumn("motor_temp", F.lit(42.0) + (F.col("id") % 100) * 0.12)
    .withColumn("battery_v",  F.lit(12.6) - (F.col("id") % 1000) * 0.0008)
    .drop("id")
    .repartition(8, "robot_id")
)

print("Partitions:", logs.rdd.getNumPartitions())
logs.printSchema()
logs.show(5, truncate=False)

Partitions: 8
root
 |-- robot_id: string (nullable = true)
 |-- sensor_id: string (nullable = true)
 |-- ts_ms: long (nullable = false)
 |-- ax: double (nullable = true)
 |-- ay: double (nullable = true)
 |-- az: double (nullable = true)
 |-- motor_temp: double (nullable = true)
 |-- battery_v: double (nullable = true)

+--------+---------+-------------+------------------+------------------+-----------------+----------+------------------+
|robot_id|sensor_id|ts_ms        |ax                |ay                |az               |motor_temp|battery_v         |
+--------+---------+-------------+------------------+------------------+-----------------+----------+------------------+
|robot-6 |imu-2    |1720000000300|0.2677026264271346|0.9787710779360422|9.817986673331747|42.72     |12.5952           |
|robot-6 |imu-2    |1720000000700|0.5011861979208834|0.9593727128473789|9.828497744450745|43.68     |12.588799999999999|
|robot-6 |imu-2    |1720000001100|0.7507388788989693|0.8525245220595057|9

## Упражнение 2.1. Кэширование

Датасет будет использоваться многократно. Закрепите его в памяти уровнем
`StorageLevel.MEMORY_AND_DISK` и **материализуйте** действием `count()` — без действия
кэш останется пустым, потому что трансформации ленивы.

In [14]:
logs.persist(StorageLevel.MEMORY_AND_DISK)
n_rows = logs.count()

print("Записей:", n_rows)
assert n_rows == N
assert logs.storageLevel.useMemory

Записей: 500000


## Упражнение 2.2. Агрегация по роботам

Постройте DataFrame `summary`, сгруппированный по `robot_id`, со столбцами:

| Столбец | Смысл |
|---|---|
| `records` | число записей |
| `avg_motor_temp` | средняя температура двигателя |
| `max_motor_temp` | максимальная температура двигателя |
| `avg_battery_v` | среднее напряжение батареи |
| `accel_rms` | среднеквадратичное ускорение |

Среднеквадратичное ускорение считается по формуле

$$\mathrm{RMS} = \sqrt{\overline{a_x^2 + a_y^2 + a_z^2}}$$

то есть **сначала усреднение суммы квадратов, потом корень**. Результат отсортируйте по
`robot_id`.

In [15]:
t0 = time.perf_counter()

summary = (
    logs.groupBy("robot_id")
    .agg(
        F.count("*").alias("records"),
        F.avg("motor_temp").alias("avg_motor_temp"),
        F.max("motor_temp").alias("max_motor_temp"),
        F.avg("battery_v").alias("avg_battery_v"),
        F.sqrt(
            F.avg(F.pow("ax", 2) + F.pow("ay", 2) + F.pow("az", 2))
        ).alias("accel_rms"),
    )
    .orderBy("robot_id")
)

summary.show(truncate=False)
print(f"Время агрегации: {time.perf_counter() - t0:.3f} с")

rows = summary.collect()
assert summary.count() == 8
assert set(summary.columns) == {"robot_id", "records", "avg_motor_temp",
                                "max_motor_temp", "avg_battery_v", "accel_rms"}
assert sum(r["records"] for r in rows) == N
assert all(9.0 < r["accel_rms"] < 11.0 for r in rows), "RMS должен быть около 9.9"

+--------+-------+------------------+------------------+------------------+-----------------+
|robot_id|records|avg_motor_temp    |max_motor_temp    |avg_battery_v     |accel_rms        |
+--------+-------+------------------+------------------+------------------+-----------------+
|robot-0 |62500  |47.76             |53.519999999999996|12.203200000000136|9.861039321834712|
|robot-1 |62500  |47.88             |53.64             |12.202400000000335|9.861039538004164|
|robot-2 |62500  |47.99999999999999 |53.76             |12.20159999999979 |9.861039687702604|
|robot-3 |62500  |48.11999999999999 |53.879999999999995|12.20079999999964 |9.861039930523152|
|robot-4 |62500  |47.75999999999999 |53.519999999999996|12.19999999999999 |9.8610400527132  |
|robot-5 |62500  |47.88             |53.64             |12.199200000000442|9.861040135458744|
|robot-6 |62500  |47.999999999999986|53.76             |12.198399999999749|9.861040336598682|
|robot-7 |62500  |48.12             |53.879999999999995|12.1

## Упражнение 2.3. Распределение записей по partition

Проверьте, как записи фактически разложены по partition. Для этого пригодится
`rdd.mapPartitionsWithIndex`: она получает индекс partition и итератор по её записям.

Ответьте: распределение равномерное? Почему получилось именно так?

In [16]:
partition_sizes = (
    logs.rdd
    .mapPartitionsWithIndex(lambda idx, it: [(idx, sum(1 for _ in it))])
    .collect()
)
print("Размеры partition:", sorted(partition_sizes))
assert sum(x[1] for x in partition_sizes) == N

ANSWER_2_3 = """
Распределение НЕравномерное, и это главный вывод упражнения. Типичная картина: две partition
пустые, одна содержит втрое больше записей, чем остальные.

Причина: repartition(8, "robot_id") раскладывает записи по хэшу от robot_id, а не по порядку.
Восемь ключей и восемь partition вовсе не гарантируют взаимно однозначного соответствия: хэш
может отправить три разных robot_id в одну корзину и оставить две другие пустыми. Записей у
каждого робота поровну (62 500), но корзины получаются разного размера.

Это и есть перекос (data skew) в чистом виде. Практическое следствие: исполнитель, которому
досталась тройная partition, работает втрое дольше, а два других простаивают — и вся стадия ждёт
самого медленного. В реальных данных перекос обычно сильнее, потому что и сами ключи распределены
неравномерно: один робот может давать половину всех событий.
"""
print(ANSWER_2_3)

Размеры partition: [(0, 62500), (1, 62500), (2, 62500), (3, 0), (4, 0), (5, 187500), (6, 62500), (7, 62500)]

Распределение НЕравномерное, и это главный вывод упражнения. Типичная картина: две partition
пустые, одна содержит втрое больше записей, чем остальные.

Причина: repartition(8, "robot_id") раскладывает записи по хэшу от robot_id, а не по порядку.
Восемь ключей и восемь partition вовсе не гарантируют взаимно однозначного соответствия: хэш
может отправить три разных robot_id в одну корзину и оставить две другие пустыми. Записей у
каждого робота поровну (62 500), но корзины получаются разного размера.

Это и есть перекос (data skew) в чистом виде. Практическое следствие: исполнитель, которому
досталась тройная partition, работает втрое дольше, а два других простаивают — и вся стадия ждёт
самого медленного. В реальных данных перекос обычно сильнее, потому что и сами ключи распределены
неравномерно: один робот может давать половину всех событий.



---
# Часть 3. Spark SQL (2 балла)

Тот же DataFrame можно запросить языком SQL. Оба API проходят через один и тот же оптимизатор
Catalyst, поэтому при одинаковой семантике планы выполнения совпадают.

## Упражнение 3.1

Зарегистрируйте `logs` как временное представление `robot_logs` и напишите SQL-запрос, который
для каждого робота возвращает число записей `n`, среднюю температуру `avg_temp` (округлить до 3
знаков), максимальную температуру `max_temp` (3 знака) и среднее напряжение `avg_voltage`
(4 знака). Оставьте только роботов с `MAX(motor_temp) > 50` и отсортируйте по `max_temp`
по убыванию.

In [19]:
logs.createOrReplaceTempView("robot_logs")

sql_result = spark.sql("""
SELECT
    robot_id,
    COUNT(*)                    AS n,
    ROUND(AVG(motor_temp), 3)   AS avg_temp,
    ROUND(MAX(motor_temp), 3)   AS max_temp,
    ROUND(AVG(battery_v), 4)    AS avg_voltage
FROM robot_logs
GROUP BY robot_id
HAVING MAX(motor_temp) > 50
ORDER BY max_temp DESC
""")

sql_result.show(truncate=False)
assert set(sql_result.columns) == {"robot_id", "n", "avg_temp", "max_temp", "avg_voltage"}
assert sql_result.count() >= 1

+--------+-----+--------+--------+-----------+
|robot_id|n    |avg_temp|max_temp|avg_voltage|
+--------+-----+--------+--------+-----------+
|robot-3 |62500|48.12   |53.88   |12.2008    |
|robot-7 |62500|48.12   |53.88   |12.1976    |
|robot-6 |62500|48.0    |53.76   |12.1984    |
|robot-2 |62500|48.0    |53.76   |12.2016    |
|robot-5 |62500|47.88   |53.64   |12.1992    |
|robot-1 |62500|47.88   |53.64   |12.2024    |
|robot-4 |62500|47.76   |53.52   |12.2       |
|robot-0 |62500|47.76   |53.52   |12.2032    |
+--------+-----+--------+--------+-----------+



## Упражнение 3.2. Сверка двух API

Убедитесь, что DataFrame API и SQL дают одинаковый результат: сравните `avg_motor_temp` из
`summary` с `avg_temp` из `sql_result` для каждого робота (с точностью до округления).

In [20]:
check = (
    summary.alias("d")
    .join(sql_result.alias("s"), "robot_id")
    .select(
        "robot_id",
        F.round(F.col("d.avg_motor_temp"), 3).alias("df_avg"),
        F.col("s.avg_temp").alias("sql_avg"),
    )
    .withColumn("ok", F.abs(F.col("df_avg") - F.col("sql_avg")) < 1e-6)
    .orderBy("robot_id")
)
check.show(truncate=False)

assert check.filter("NOT ok").count() == 0, "Результаты двух API разошлись"

+--------+------+-------+----+
|robot_id|df_avg|sql_avg|ok  |
+--------+------+-------+----+
|robot-0 |47.76 |47.76  |true|
|robot-1 |47.88 |47.88  |true|
|robot-2 |48.0  |48.0   |true|
|robot-3 |48.12 |48.12  |true|
|robot-4 |47.76 |47.76  |true|
|robot-5 |47.88 |47.88  |true|
|robot-6 |48.0  |48.0   |true|
|robot-7 |48.12 |48.12  |true|
+--------+------+-------+----+



---
# Часть 4. Physical plan и партиционирование (2 балла)

## Упражнение 4.1. Чтение плана выполнения

Выполните `explain(mode="formatted")` для `summary`. Найдите в физическом плане операторы
`Exchange` и `HashAggregate` и объясните, что каждый из них делает.

In [ ]:
summary.explain(mode="formatted")

In [22]:
ANSWER_4_1 = """
В плане видны два HashAggregate и один Exchange между ними — это классическая двухфазная
агрегация.

Первый (нижний) HashAggregate выполняет частичную агрегацию: каждая partition независимо считает
свои частичные count, sum и max. Это локальная операция, данные никуда не пересылаются.

Exchange hashpartitioning(robot_id, ...) — это shuffle. Он перераспределяет частичные результаты
по узлам так, чтобы все записи с одним robot_id оказались в одной partition. Единственный шаг,
где данные идут по сети; он же образует границу stage.

Второй (верхний) HashAggregate доводит агрегацию до конца: складывает частичные суммы и счётчики,
берёт максимум из максимумов. Это возможно, потому что count, sum и max ассоциативны.

Дополнительно виден Exchange rangepartitioning для ORDER BY: чтобы отсортировать результат
глобально, Spark разбивает диапазон значений ключа между partition. AdaptiveSparkPlan сверху
означает, что включён AQE и план может быть уточнён по фактической статистике во время выполнения.
"""
print(ANSWER_4_1)


В плане видны два HashAggregate и один Exchange между ними — это классическая двухфазная
агрегация.

Первый (нижний) HashAggregate выполняет частичную агрегацию: каждая partition независимо считает
свои частичные count, sum и max. Это локальная операция, данные никуда не пересылаются.

Exchange hashpartitioning(robot_id, ...) — это shuffle. Он перераспределяет частичные результаты
по узлам так, чтобы все записи с одним robot_id оказались в одной partition. Единственный шаг,
где данные идут по сети; он же образует границу stage.

Второй (верхний) HashAggregate доводит агрегацию до конца: складывает частичные суммы и счётчики,
берёт максимум из максимумов. Это возможно, потому что count, sum и max ассоциативны.

Дополнительно виден Exchange rangepartitioning для ORDER BY: чтобы отсортировать результат
глобально, Spark разбивает диапазон значений ключа между partition. AdaptiveSparkPlan сверху
означает, что включён AQE и план может быть уточнён по фактической статистике во время выполнен

## Упражнение 4.2. Эксперимент с числом partition

Измерьте время выполнения агрегации при `spark.sql.shuffle.partitions`, равном 2, 4, 8 и 64,
на одном и том же runtime. Для каждого значения:

1. установите параметр через `spark.conf.set(...)`;
2. выполните агрегацию и **обязательно** вызовите действие (например, `collect()`), иначе
   из-за ленивости ничего не посчитается;
3. запишите время в словарь `timings`.

Каждое измерение повторите 3 раза и возьмите минимум — так меньше влияет разогрев JVM.

> **Важно.** На время эксперимента отключите Adaptive Query Execution:
> `spark.conf.set("spark.sql.adaptive.enabled", "false")`. Иначе AQE сам объединит лишние
> partition после shuffle, и влияние параметра будет незаметно — вы измерите работу оптимизатора,
> а не то, что хотели. После эксперимента верните `"true"`.

In [23]:
def run_aggregation():
    """Выполняет ту же агрегацию и возвращает результат (действие обязательно)."""
    return (
        logs.groupBy("robot_id")
        .agg(F.count("*").alias("records"),
             F.avg("motor_temp").alias("avg_motor_temp"),
             F.max("motor_temp").alias("max_motor_temp"))
        .collect()
    )


spark.conf.set("spark.sql.adaptive.enabled", "false")   # чистый эксперимент

timings = {}
for p in [2, 4, 8, 64]:
    spark.conf.set("spark.sql.shuffle.partitions", str(p))
    best = float("inf")
    for _ in range(3):
        t0 = time.perf_counter()
        run_aggregation()
        best = min(best, time.perf_counter() - t0)
    timings[p] = best

spark.conf.set("spark.sql.adaptive.enabled", "true")

for p, t in sorted(timings.items()):
    print(f"shuffle.partitions = {p:>3} -> {t:.3f} с")

assert set(timings) == {2, 4, 8, 64}

shuffle.partitions =   2 -> 0.554 с
shuffle.partitions =   4 -> 0.367 с
shuffle.partitions =   8 -> 0.754 с
shuffle.partitions =  64 -> 0.399 с


## Упражнение 4.3. Интерпретация

Ответьте письменно:

1. Какое число partition оказалось лучшим и почему время не убывает монотонно с ростом partition?
2. Что произойдёт при `spark.sql.shuffle.partitions = 200` (значение по умолчанию) на этих данных?
3. Как связано число partition с числом ядер, доступных Colab (`sc.defaultParallelism`)?

In [24]:
ANSWER_4_3 = """
1. Лучшим оказывается число partition, близкое к числу доступных ядер. При двух partition
   параллелизм недоиспользован: часть ядер простаивает. При 64 partition каждая задача
   обрабатывает крошечную порцию данных, и накладные расходы на планирование, запуск задач,
   сериализацию и сбор результатов начинают превышать полезную работу. Поэтому зависимость
   немонотонна: сначала время падает, затем растёт.

   Честная оговорка по фактическим цифрам: на 500 тысячах записей и восьми ключах разброс между
   вариантами составляет доли секунды и сопоставим с шумом измерения. Данных слишком мало, чтобы
   эффект проявился в полную силу; на десятках миллионов записей разница становится кратной.
   Именно поэтому на время эксперимента и отключается AQE — с ним Spark сам сводит лишние
   partition к разумному числу, и различие исчезает совсем.

2. При 200 partition на 500 тысячах записей и восьми ключах картина ухудшается ещё сильнее:
   Spark создаст 200 задач, из которых осмысленную работу выполнят не более восьми (по числу
   различных robot_id), а остальные окажутся пустыми. Всё время уйдёт на оркестрацию. Значение
   200 по умолчанию рассчитано на промышленные объёмы данных, а не на учебные.

3. Разумная эвристика: число partition кратно числу ядер, обычно от одного до четырёх на ядро.
   Меньше ядер — часть ресурсов простаивает; сильно больше — растут накладные расходы. Полезно
   ориентироваться на sc.defaultParallelism и на целевой размер partition порядка 100–200 МБ.
   Именно эту задачу и решает автоматически Adaptive Query Execution, объединяя мелкие partition
   после shuffle.
"""
print(ANSWER_4_3)


1. Лучшим оказывается число partition, близкое к числу доступных ядер. При двух partition
   параллелизм недоиспользован: часть ядер простаивает. При 64 partition каждая задача
   обрабатывает крошечную порцию данных, и накладные расходы на планирование, запуск задач,
   сериализацию и сбор результатов начинают превышать полезную работу. Поэтому зависимость
   немонотонна: сначала время падает, затем растёт.

   Честная оговорка по фактическим цифрам: на 500 тысячах записей и восьми ключах разброс между
   вариантами составляет доли секунды и сопоставим с шумом измерения. Данных слишком мало, чтобы
   эффект проявился в полную силу; на десятках миллионов записей разница становится кратной.
   Именно поэтому на время эксперимента и отключается AQE — с ним Spark сам сводит лишние
   partition к разумному числу, и различие исчезает совсем.

2. При 200 partition на 500 тысячах записей и восьми ключах картина ухудшается ещё сильнее:
   Spark создаст 200 задач, из которых осмысленную работу

In [25]:
# Возвращаем исходное значение и освобождаем кэш
spark.conf.set("spark.sql.shuffle.partitions", "8")
logs.unpersist()
print("Готово. Часть 4 завершена.")

Готово. Часть 4 завершена.


---
# Часть 5. TEACH CARD (2 балла)

Заполните паспорт педагогической адаптации выполненной инженерной задачи. Это **обязательная**
часть работы: она проверяет, поняли ли вы суть настолько, чтобы объяснить её школьнику.

**Требования к заполнению.** Формулировки конкретные, а не общие. «Расскажу детям про Spark» —
не засчитывается. Нужны: измеримый результат, критерий успеха и честно названные ограничения.

| Поле | Содержание |
|---|---|
| **Название учебного проекта** | Чей робот самый горячий? Анализ телеметрии роя роботов |
| **Целевая аудитория** | кружок программирования для 10-11 классов |
| **Исследовательский вопрос** | Как найти робота с самой высокой температурой в рое из 100 машин, если их логи не помещаются в оперативную память одного компьютера и требуют долгой обработки? |
| **Источник данных** | Симулированный CSV-файл на 1 млн строк. Генерируется скриптом на Python с фиксированным random.seed(42), чтобы датасет был идентичным у всех обучающихся. |
| **Инженерная концепция, сохраняемая без упрощения** | Идея того, что данные нужно дробить на независимые куски (Map), обрабатывать параллельно и сводить (Reduce). |
| **Что упрощается относительно университетской лабораторной** | Убираем настоящий кластер, Spark и JVM. Используем чистый Python и стандартную библиотеку concurrent.futures для имитации распределённых узлов на одном ПК. |
| **Алгоритм действий обучающегося (4–6 шагов)** | 1. Сгенерировать CSV на 1 млн строк (с фиксированным seed). 2. Написать функцию чтения файла в 1 поток (получить обычный ответ). 3. Написать функцию map_chunk(): читает файл кусками, находит MAX(Temp) для каждого ID. 4. Запустить map_chunk() через ProcessPoolExecutor (имитация кластера, то есть разделения на потоки). 5. Сравнить время выполнения: 1 поток против N потоков (рассчитать ускорение по выведенной формуле). |
| **Измеримый результат / метрика** | 1. Корректно определённый ID робота с максимальной температурой (совпадает с эталоном). 2. Коэффициент ускорения: во сколько раз многопоточный код быстрее однопоточного. |
| **Критерий успешного выполнения** | Код работает без ошибок. Функции Map и Reduce написаны раздельно. Результаты многопоточного кода совпадают с однопоточным. |
| **Вариант усложнения для НТО / хакатона** | Добавить «сломанный датчик»: в 1% строк температура равна NaN или строке "ERR". Задание: модифицировать map_chunk, чтобы использовать try/except и фильтровать мусор до стадии Reduce. |
| **Риски и ограничения** | 1. если файл маленький (100 строк), многопоточность окажется медленнее из-за затрат на создание процессов. 2. одна битая строка в CSV может «уронить» весь пул процессов, если не предусмотрена обработка ошибок. |

---
## Итоговая самопроверка

Ячейка ниже проверяет, что все обязательные объекты созданы и задания выполнены.

In [26]:
checks = {
    "1.1 mapper":           len(pairs) > 40_000,
    "1.2 partitioner":      len(grouped) == len(set(w for w, _ in pairs)),
    "1.3 reducer":          sum(counts.values()) == len(pairs),
    "1.4 RDD word count":   top5_rdd == top5,
    "1.5 ответ":            "TODO" not in ANSWER_1_5,
    "2.1 кэш":              n_rows == N,
    "2.2 агрегация":        summary.count() == 8,
    "2.3 partition":        sum(x[1] for x in partition_sizes) == N and "TODO" not in ANSWER_2_3,
    "3.1 Spark SQL":        sql_result.count() >= 1,
    "3.2 сверка API":       check.filter("NOT ok").count() == 0,
    "4.1 physical plan":    "TODO" not in ANSWER_4_1,
    "4.2 бенчмарк":         set(timings) == {2, 4, 8, 64},
    "4.3 интерпретация":    "TODO" not in ANSWER_4_3,
}

for name, ok in checks.items():
    print(f"{'OK ' if ok else 'НЕТ'}  {name}")

print()
print(f"Выполнено: {sum(checks.values())} из {len(checks)}")
print("TEACH CARD проверяется преподавателем вручную.")

OK   1.1 mapper
OK   1.2 partitioner
OK   1.3 reducer
OK   1.4 RDD word count
OK   1.5 ответ
OK   2.1 кэш
OK   2.2 агрегация
OK   2.3 partition
OK   3.1 Spark SQL
OK   3.2 сверка API
OK   4.1 physical plan
OK   4.2 бенчмарк
OK   4.3 интерпретация

Выполнено: 13 из 13
TEACH CARD проверяется преподавателем вручную.


In [27]:
spark.stop()
print("SparkSession остановлена.")

SparkSession остановлена.
